# 实验三：视频质量评估——PSNR 与 SSIM 的较量（Video Quality Assessment）
## FMI Course · Kaggle Hands-on Lab

**课程**：未来媒体互联网（Future Media & Internet）&nbsp;|&nbsp; **预计时长**：~10 分钟 &nbsp;|&nbsp; **运行环境**：Kaggle Notebook（CPU）&nbsp;|&nbsp; **无需 GPU**

---

## 实验概述

视频质量评估是编码优化和传输质量监控的基础：无论是压缩算法调参、码率分配决策，还是网络传输质量监控，都需要一个客观指标来衡量「视频质量到底有多好」。业界最经典的指标是 PSNR（峰值信噪比，Peak Signal-to-Noise Ratio），计算简单、历史悠久；但近十几年来，YouTube、Netflix 等平台越来越依赖 SSIM（结构相似性，Structural Similarity）以及更先进的感知指标。

本实验通过对同一张测试图施加从轻微到严重的压缩失真，同时计算 PSNR 和 SSIM 两种指标，直观揭示两者为什么会给出不一致的排名——以及为什么「像素误差小」不等于「人眼觉得好看」。

本实验对应课程中「视频质量评估与感知指标」部分的核心内容。

## 学习目标

完成本实验后，你应该能够：

1. 写出 PSNR 的计算公式，并解释它衡量的是「像素级误差」；
2. 说明 SSIM 由亮度、对比度、结构三个相似性分量组成，衡量的是「结构/感知相似度」；
3. 解释「PSNR 高但视觉观感差」这种现象可能出现的场景及原因；
4. 从实验结果中判断哪种失真类型下 PSNR 与 SSIM 的排名最容易出现分歧；
5. 说明为什么 Netflix、YouTube 等平台在编码优化中更依赖感知指标而非纯 PSNR。

## 背景与基本原理

### PSNR（峰值信噪比）：基于像素误差

PSNR 的核心是均方误差（Mean Squared Error, MSE）——逐像素比较原图与失真图之间的差值平方，再取平均：

`PSNR = 20 · log₁₀(MAX / √MSE)`

其中 `MAX` 是像素最大取值（本实验中归一化为 1.0）。**PSNR 的单位是 dB，数值越高代表失真越小。**

PSNR 的局限性在于：它只关心「每个像素差了多少」，完全不关心这些差异在空间上是如何分布的。举例来说，把一整块区域的像素值统一偏移一点点（视觉上几乎看不出来），和把同样数量的像素误差集中破坏图像中的一条关键边缘（视觉上非常明显），只要 MSE 相同，PSNR 就完全相同——但人眼的感受天差地别。

### SSIM（结构相似性）：模拟人眼感知

SSIM 由三个相似性分量的乘积构成：

| 分量 | 衡量内容 |
|------|---------|
| 亮度相似性（Luminance） | 两幅图像的平均亮度是否接近 |
| 对比度相似性（Contrast） | 两幅图像的亮度方差（对比度）是否接近 |
| 结构相似性（Structure） | 两幅图像去除亮度和对比度影响后，剩余的结构模式（边缘、纹理）是否接近 |

SSIM 的取值范围是 **0 到 1**，越接近 1 表示与原图越相似。它通过局部窗口（本实验采用 11×11 滑动窗口）计算，因此对**局部结构破坏**（如方块效应、边缘模糊）比 PSNR 更敏感。

### 为什么「PSNR 高但看起来差」？

以 JPEG 常见的**方块效应（Blocking Artifacts）**为例：块内像素被替换为均值，块边界处产生突兀的不连续。这种失真造成的均方误差可能并不大（因为块内像素本身变化就不剧烈），但块边界处的结构断裂在视觉上非常突兀、容易被人眼捕捉——这正是 SSIM 相比 PSNR 更能反映真实观感的原因。

## 实验设计

**测试图像**：程序生成的合成灰度图（256×256），刻意包含三类内容以覆盖不同的失真敏感区域——白色方块（模拟文字/图标的锐利边缘）、渐变背景（低频平滑区域）、密集细线（高频纹理细节）。

**失真级别**：共 4 档——`Original`（无损）、`Light`（轻微：仅加少量高斯噪声）、`Medium`（中等：噪声 + 8×8 块效应）、`Heavy`（严重：更强噪声 + 16×16 块效应）。块效应模拟的是 JPEG 类编码在低码率下的典型失真模式。

**评价指标**：对每个失真级别，同时计算 PSNR（自定义实现）与 SSIM（基于 PyTorch 张量运算的简化实现，用 11×11 局部窗口估计局部统计量）。

**预期观察**：PSNR 和 SSIM 都应随失真程度增加而下降，但下降的「节奏」不同——在引入块效应的中/重度失真处，SSIM 的下降幅度往往比 PSNR 更剧烈，因为块效应对结构相似性的破坏比对像素均方误差的影响更显著。

## 运行环境说明

| 项目 | 说明 |
|------|------|
| 运行环境 | Kaggle Notebook |
| 计算资源 | CPU（无需 GPU） |
| 网络访问 | 不需要（Internet Off） |
| 主要依赖 | PyTorch、PIL、NumPy、Matplotlib（Kaggle 已预装） |

直接点击「Run All」即可运行全部实验，无需上传数据或安装额外包。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import requests
from io import BytesIO
import torch
import torch.nn.functional as F

print("环境就绪 ✅")

## 步骤一：生成测试图像并模拟不同程度的压缩失真

下面的代码做两件事：

**① 构建合成测试图 `create_test_image()`**：包含白色方块（模拟文字/图标边缘）、渐变背景（低频区域）、密集细线（高频细节）——这三类内容分别对应实际视频画面中最容易受到不同类型失真影响的区域。

**② 模拟失真 `degrade_quality()`**：对图像叠加高斯噪声，并在中/重度级别下额外引入块效应（将图像分块后用块内均值替换，模拟 JPEG 类编码在低码率下的典型失真）。

运行后会看到 4 张图并排展示：`Original / Light / Medium / Heavy`。

**观察重点**：留意 `Heavy` 版本中细线区域和白色方块边缘的块状痕迹，这正是后续 SSIM 会重点捕捉的结构性失真。

In [ ]:
# 用内置图案模拟"Original图 vs 压缩图"的对比
# 真实场景中需准备视频帧，这里用合成图演示原理

def create_test_image():
    """创建包含细节的测试图：文字 + 渐变 + 边缘"""
    img = np.ones((256, 256), dtype=np.float32) * 0.5
    # 白色方块（模拟文字）
    img[60:100, 40:200] = 0.9
    img[120:160, 40:200] = 0.9
    # 渐变背景
    for i in range(256):
        img[i, 210:256] = i / 256
    # 细线（高频细节）
    for j in range(180, 190):
        img[:, j] = 1.0
    return img

def degrade_quality(img, level):
    """模拟不同程度的压缩失真"""
    # level 0=无损, 1=轻微, 2=中等, 3=严重
    noise_levels = [0.0, 0.03, 0.1, 0.25]
    blur_levels = [0, 1, 3, 7]
    
    degraded = img.copy()
    # 加噪
    noise = np.random.randn(*img.shape) * noise_levels[level]
    degraded = degraded + noise
    # 模拟块效应（JPEG 风格失真）
    if level >= 2:
        block_size = 8 if level == 2 else 16
        for i in range(0, 256, block_size):
            for j in range(0, 256, block_size):
                degraded[i:i+block_size, j:j+block_size] = degraded[i:i+block_size, j:j+block_size].mean()
    degraded = np.clip(degraded, 0, 1)
    return degraded

original = create_test_image()
versions = [degrade_quality(original, lv) for lv in range(4)]

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
titles = ['Original', 'Light', 'Medium', 'Heavy']
for i, (ax, img, title) in enumerate(zip(axes, versions, titles)):
    ax.imshow(img, cmap='gray', vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis('off')
plt.suptitle('Images at Different Compression Levels')
plt.show()

## 步骤二：计算 PSNR 与 SSIM 指标

这部分代码实现并计算两个指标：

- **`psnr()`**：直接套用 PSNR 公式，基于原图与失真图之间的均方误差（MSE）计算。
- **`ssim_torch()`**：用 PyTorch 的 `avg_pool2d` 实现 11×11 局部滑动窗口，分别估计局部均值、局部方差、局部协方差，再代入 SSIM 公式的亮度/对比度/结构三项，最终取全图平均值。

代码会依次打印每个失真级别对应的 PSNR（单位 dB）与 SSIM（0~1）数值。

> 思考：仅从数值上看，你能预判哪个失真级别下两个指标的「排名一致性」最容易出现分歧吗？

In [ ]:
# Calculate quality metrics
def psnr(original, degraded):
    mse = np.mean((original - degraded) ** 2)
    if mse < 1e-10:
        return 100.0
    return float(20 * np.log10(1.0 / np.sqrt(mse)))

def ssim_torch(original, degraded):
    """Robust SSIM with numerical safeguards"""
    orig_t = torch.tensor(original).unsqueeze(0).unsqueeze(0).float()
    deg_t = torch.tensor(degraded).unsqueeze(0).unsqueeze(0).float()
    
    # Local means via 11x11 box filter
    mu_x = F.avg_pool2d(orig_t, 11, 1, 5)
    mu_y = F.avg_pool2d(deg_t, 11, 1, 5)
    
    # Local variances (clamp to prevent negative from FP errors)
    sigma_x = torch.clamp(F.avg_pool2d(orig_t ** 2, 11, 1, 5) - mu_x ** 2, min=0.0)
    sigma_y = torch.clamp(F.avg_pool2d(deg_t ** 2, 11, 1, 5) - mu_y ** 2, min=0.0)
    sigma_xy = F.avg_pool2d(orig_t * deg_t, 11, 1, 5) - mu_x * mu_y
    
    C1 = 0.01 ** 2
    C2 = 0.03 ** 2
    num = (2 * mu_x * mu_y + C1) * (2 * sigma_xy + C2)
    den = (mu_x ** 2 + mu_y ** 2 + C1) * (sigma_x + sigma_y + C2)
    ssim_map = num / den
    return ssim_map.mean().item()

# Compute all versions
results = []
for i, (ver, name) in enumerate(zip(versions, titles)):
    p = psnr(original, ver)
    s = ssim_torch(original, ver)
    results.append(dict(name=name, PSNR=p, SSIM=s))
    print(f"{name:8s} | PSNR: {p:5.1f} dB | SSIM: {s:.3f}")

print("\nKey observation:")
print("PSNR is pixel-based. SSIM mimics human perception (structure and texture).")
print("When PSNR values are similar, SSIM can differ greatly - humans notice structural distortion more.")


## 步骤三：可视化对比两个指标随失真程度的变化趋势

下方代码将 PSNR 与 SSIM 分别绘制为柱状图，便于横向比较四个失真级别下两个指标的变化节奏是否同步。

**观察重点**：对比左右两张柱状图的下降曲线形状——如果 PSNR 近似线性下降，而 SSIM 在引入块效应（Medium/Heavy）的位置出现更陡峭的跳水，就说明 SSIM 对结构性失真（块效应）比 PSNR 更敏感。

In [ ]:
# 可视化对比
names = [r['name'] for r in results]
psnrs = [r['PSNR'] for r in results]
ssims = [r['SSIM'] for r in results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

colors = ['#2ecc71', '#f39c12', '#e67e22', '#e74c3c']
ax1.bar(names, psnrs, color=colors)
ax1.set_ylabel('PSNR (dB)')
ax1.set_title('Traditional: PSNR (higher=better)')
for i, v in enumerate(psnrs):
    ax1.text(i, v + 0.5, f'{v:.1f}', ha='center', fontweight='bold')

ax2.bar(names, ssims, color=colors)
ax2.set_ylabel('SSIM')
ax2.set_title('Perceptual: SSIM (closer to 1=better)')
for i, v in enumerate(ssims):
    ax2.text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

plt.suptitle('Traditional vs Perceptual Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n课堂互动问题：")
print("1. PSNR 高但看起来差的情况在什么场景最常见？（提示：图像有纹理的区域）")
print("2. 为什么 YouTube 不用 PSNR 来决定编码参数？")
print("3. AI 质量评估模型（如 LPIPS）比 SSIM 好在哪？（提示：用神经网络学习「人觉得好不好看」）")

## 实验结果与分析

### 指标变化趋势

运行实验后，你通常会观察到：

- **PSNR** 随失真级别单调下降，且下降趋势相对平缓、接近线性——因为它只累计像素级别的平方误差，噪声和块效应对 MSE 的贡献是「叠加式」的。
- **SSIM** 在 `Light`（仅噪声，无块效应）阶段下降相对温和，但一旦进入 `Medium`/`Heavy`（引入块效应）阶段，下降幅度往往明显更陡——块边界处结构的不连续性对局部窗口内的协方差/方差估计影响很大，而这正是 SSIM 结构分量重点捕捉的内容。

### 两者不一致的原因

PSNR 和 SSIM 的分歧集中体现在**块效应引入的阶段**：两者的 MSE 增量可能相近，但 SSIM 认为块效应对图像「结构」的破坏远比同等 MSE 的随机噪声更严重。这正好印证了背景原理中提到的核心观点——**局部结构是否连贯，比像素误差的绝对大小更接近人类视觉感知**。

### 实验局限性

需要注意：本实验的 SSIM 实现是一个简化版本（未做多尺度、未使用真实自然图像的统计特性验证），仅用于演示原理。真实场景中，Netflix 等平台使用的感知指标（如 VMAF）会融合更多视觉感知模型和大规模主观打分数据训练得到，比单一的 SSIM 更贴近真实观感。

## 从实验到实际系统

本实验演示了 PSNR 与 SSIM 的核心差异，但真实的视频质量评估体系更加丰富：

- **Netflix 的 VMAF（Video Multimethod Assessment Fusion）**：融合多种感知特征（包括 SSIM 的变体、细节损失度量等），并用机器学习模型基于大规模人工主观打分数据训练得到，是目前工业界公认更贴近人眼感知的质量指标；
- **Google 的 LPIPS（Learned Perceptual Image Patch Similarity）**：完全基于深度神经网络特征空间中的距离衡量图像相似度，能捕捉传统指标难以刻画的语义级感知差异；
- **DISTS**：结合了结构相似性和纹理相似性，专门针对纹理丰富区域做了优化；
- **为什么感知指标能节省带宽**：如果编码器以 PSNR 为优化目标，可能会「浪费」码率在人眼不敏感的区域；而以感知指标为目标进行码率分配，可以把有限的码率优先投入到人眼真正敏感的区域，在相同主观质量下降低整体码率。

---

## 本实验小结

通过本实验，你应该掌握以下核心结论：

1. **PSNR 衡量像素误差，简单但不完全贴合人眼感知**：基于逐像素均方误差，计算高效但忽略了误差的空间分布。
2. **SSIM 衡量结构相似性，更贴近人类视觉系统**：通过局部窗口比较亮度、对比度、结构三个维度。
3. **两者在块效应等结构性失真下容易出现分歧**：结构破坏对 SSIM 的影响远大于对 PSNR 的影响。
4. **单一指标不能完全代表主观质量**：工业界通常会结合多种指标或使用融合多种特征的高级指标（如 VMAF）。
5. **指标选择直接影响编码优化方向**：以感知指标为优化目标，能在相同主观质量下实现更高的压缩效率。

---

## 思考与拓展

以下问题没有唯一答案，鼓励你修改代码并重新运行：

1. **换用真实图片**：用 PIL 读取一张真实照片替换 `create_test_image()` 生成的合成图，重复本实验的对比流程，观察结论是否依然成立。
2. **实现简化版 LPIPS**：尝试用一个预训练的小型卷积网络提取特征，计算特征空间距离作为感知指标，并与 PSNR/SSIM 的排名做对比。
3. **修改失真类型**：将 `degrade_quality()` 改为模拟颜色偏移（如整体调整色调）而非噪声+块效应，观察两个指标此时的反应是否与本实验一致。
4. **调整 SSIM 窗口大小**：把 `ssim_torch()` 中的 11×11 窗口改为 5×5 或 21×21，观察窗口大小如何影响 SSIM 对局部失真的敏感度。
5. **量化「分歧程度」**：计算每个失真级别下 PSNR 排名与 SSIM 排名的相对差异（如排名的名次差），用数值方式验证哪个阶段分歧最大。

---

← [实验二：自适应视频流与 QoE 优化](https://www.kaggle.com/code/guopingtan/fmi-demo2-qoe-optimization) &nbsp;|&nbsp; 🏠 [课程主页 · Course Home](https://www.kaggle.com/code/guopingtan/fmi-course-kaggle-hands-on-lab-start-here) &nbsp;|&nbsp; [实验四：神经压缩 vs JPEG →](https://www.kaggle.com/code/guopingtan/fmi-demo4-neural-compression)

**FMI Course · Kaggle Hands-on Lab** &nbsp;|&nbsp; MV-AI Lab · Hohai University